# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

I picked two findings from the paper (`docs/flyrank-seo-research-march-2026.pdf`) that I can *act on* — and for each I asked: where does the label come from, and does the validation design carry the claim?

**Finding A — "The Freshness Multiplier" (Finding #4, p.9, CONFIRMED)**
- **Label:** growth-to-decline ratio per freshness window (pages classified growing vs declining by 30d-vs-prev-30d impression change, cut at ±10%), plus before/after health and impressions for refreshed 365+ content.
- **The claim:** the 31–90 day freshness band is the strongest stable multiplier at **7.88:1**; the 361+ tail spikes to **283:1**; refreshed 365+ content shows **3.2x health** (10.7 → 34.5) and **57x impressions** (71 → 4,039).
- **Where the label comes from:** trend direction from GSC impressions — the same ±10% hairline my own `is_declining` label draws.
- **My methodology question:** the 361+ spike (283:1) sits on **1 declining page** (283 growing vs 1 declining). The paper itself flags that bucket as "too small and too unstable to treat as a headline multiplier" — yet the same tiny bucket feeds the 57x / 3.2x refresh claim. Does a validation design whose headline rests on a denominator of one carry a 283:1 or 57x ratio? Constructive reading: the ratio is a fluke of the small sample; the actionable claim should rest on the large 31–90 day window, not the 365+ spike.

**Finding B — "What Predicts Growth?" (ML Appendix, p.29)**
- **Label:** binary growing vs declining, cut at ±10% on 30d-vs-prev-30d impression change (p.5: "Up: >10% growth. Down: >10% decline. Stable: within +/-10%").
- **The claim:** logistic regression reaches **71% holdout accuracy** separating growing from declining pages.
- **Where the label comes from:** the same thresholded trend direction; the ML work is explicitly exploratory (80/20 split, and p.36 states "No p-values or confidence intervals are reported").
- **My methodology question:** the 10% line is a hairline — a page at +9.9% is "stable", a page at +10.1% is "growing", and the two are nearly identical. Is 71% holdout accuracy a property of the signal or of where the line was drawn? If the threshold moves (±5%, ±15%), does the accuracy survive? Constructive reading: accuracy measured on a thresholded label is partly measuring the cut itself; the model can look strong while the underlying ranking is only slightly better than chance near the boundary.

Why these two: both are CONFIRMED findings with big, actionable numbers, and both rest on validation choices I now recognise in my own Week-5 model — a hairline-threshold label, and a split that never sees a future period. They are the reason this audit exists.

In [1]:
# Section 1 — verify the two findings' numbers directly against the paper PDF
import os
from pypdf import PdfReader

REPO_ROOT = os.getcwd()
while not os.path.exists(os.path.join(REPO_ROOT, 'AGENTS.md')) and os.path.dirname(REPO_ROOT) != REPO_ROOT:
    REPO_ROOT = os.path.dirname(REPO_ROOT)

pdf = PdfReader(os.path.join(REPO_ROOT, 'docs/flyrank-seo-research-march-2026.pdf'))
p5 = pdf.pages[4].extract_text()   # p.5: trend-direction label definition
p9 = pdf.pages[8].extract_text()   # p.9: Finding #4, Freshness Multiplier
p29 = pdf.pages[28].extract_text() # p.29: ML appendix, growth prediction

checks = {
    '31-90d growth-to-decline ratio 7.88:1': '7.88:1' in p9,
    '181-360 stale bucket 3.13:1': '3.13:1' in p9,
    '361+ spike 283:1': '283:1' in p9,
    '361+ has 1 declining page': '1 declining page' in p9,
    'refresh health 10.7 -> 34.5': ('10.7' in p9 and '34.5' in p9),
    'refresh impressions 71 -> 4039': ('71' in p9 and '4039' in p9),
    '3.2x health boost': '3.2x' in p9,
    '57x impression boost': '57x' in p9,
    'ML appendix 71% holdout accuracy': '71%' in p29,
    'trend label cut at >10% / within +/-10%': ('>10%' in p5 and '+/-10%' in p5),
}
for label, ok in checks.items():
    print(f'{label:42s} {"found" if ok else "MISSING"}')
print()
assert all(checks.values()), 'paper extraction did not find every quoted number'
print('All numbers above are quoted from the paper (paper-reported). They are NOT recomputed here —')
print('the code cell only confirms the quotes exist in the PDF before I argue about them.')

31-90d growth-to-decline ratio 7.88:1      found
181-360 stale bucket 3.13:1                found
361+ spike 283:1                           found
361+ has 1 declining page                  found
refresh health 10.7 -> 34.5                found
refresh impressions 71 -> 4039             found
3.2x health boost                          found
57x impression boost                       found
ML appendix 71% holdout accuracy           found
trend label cut at >10% / within +/-10%    found

All numbers above are quoted from the paper (paper-reported). They are NOT recomputed here —
the code cell only confirms the quotes exist in the PDF before I argue about them.


## 2. My model under an honest split (before/after)

Week-5's split grouped pages by client, so the model was judged on clients it never met — right for *"would this work for a client we have not met yet?"* But train and test shared the **same calendar window** (Jan–Feb features → March label). The model was never tested on a period it had not trained on — the exact thing a reviewer needs it to do next month.

So the *after* adds **time**: train the same forest on an earlier window (December features → January label), then test on the same March rows Week-5 used. Same metric (precision@K), same K, same tie policy, same label definition.

- **Before** (Week-5, grouped by client, same window): mean fold P@50 = **0.676** (committed receipt); recomputed today on the current warehouse snapshot ≈ **0.67**. The small gap is warehouse drift in `dim_content` (content age) between the August 7 run and today, not a code change — folds and base rates are identical.
- **After** (time-aware, Dec→Jan trains, March tests): all clients tested at once gives **P@50 ≈ 0.1–0.2** — *below* the 0.25 base rate (the top-50 under this model is only ~10–16% declining), so the ranking actively reverses. Keeping Week-5's per-fold protocol (train earlier period on 4 folds, test the held-out fold's March rows) gives **mean fold P@50 ≈ 0.35–0.40**.
- **Robustness:** a second earlier training window (Nov→Dec) also collapses (P@50 ≈ 0.26–0.30 vs 0.67), so the collapse is not a December holiday artifact.

**Why it collapses:** the decline signal is not stable across months — feature–label correlations flip sign between windows (e.g. `log_imp_prev30` vs `is_declining` is +0.06 in Dec→Jan but −0.09 in the March window). A model trained on the earlier window leans on a pattern the next window contradicts. Week-5's split could not see this, because train and test shared the same period.

**Note on the time split:** the same pages appear in both windows — their January outcomes are training labels, their March outcomes never are. The test is whether the pattern transfers across time, which is exactly the deployment question.

**Measured reading:** the Week-5 0.67 is real for that one window, but it does not transfer forward in time. Under the honest split it drops to ≈0.35–0.40 (unseen clients) or ≈0.1–0.2 (unseen time). My model is a same-window ranking tool, not a forward predictor — and that is now a measured result, not a guess.

In [2]:
# Section 2 — honest before/after: Week-5 grouped split vs a time-aware split
import os, hashlib, json
import pandas as pd, numpy as np
import duckdb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

HF_TOKEN = os.environ.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REPO_ROOT = os.getcwd()
while not os.path.exists(os.path.join(REPO_ROOT, 'AGENTS.md')) and os.path.dirname(REPO_ROOT) != REPO_ROOT:
    REPO_ROOT = os.path.dirname(REPO_ROOT)
OUT_DIR = os.path.join(REPO_ROOT, 'work', 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)

REL = 'hf://datasets/FlyRank/internship-warehouse'
FM26 = lambda m: f"read_parquet('{REL}/fact_content_daily_performance/month=2026-{m}/**/*.parquet')"
FM25 = lambda m: f"read_parquet('{REL}/fact_content_daily_performance/month=2025-{m}/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

def build_frame(prev_start, prev_end, label_start, label_end, decision_date, months25, months26):
    froms = [f"{FM25(m)}" for m in months25] + [f"{FM26(m)}" for m in months26]
    src = ' UNION ALL '.join(f"SELECT * FROM {x}" for x in froms)
    q = f"""
        WITH per_content AS (
            SELECT
                f.content_hash_id, f.client_hash_id,
                SUM(CASE WHEN f.report_date >= DATE '{prev_start}' AND f.report_date < DATE '{prev_end}' THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
                SUM(CASE WHEN f.report_date >= DATE '{prev_start}' AND f.report_date < DATE '{prev_end}' THEN f.gsc_clicks ELSE 0 END) AS clk_prev30,
                AVG(CASE WHEN f.report_date >= DATE '{prev_start}' AND f.report_date < DATE '{prev_end}' THEN f.gsc_avg_position END) AS pos_prev30,
                COUNT(DISTINCT CASE WHEN f.report_date >= DATE '{prev_start}' AND f.report_date < DATE '{prev_end}' AND f.gsc_impressions > 0 THEN f.report_date END) AS days_with_imp_prev30,
                SUM(CASE WHEN f.report_date >= DATE '{label_start}' AND f.report_date < DATE '{label_end}' THEN f.gsc_impressions ELSE 0 END) AS imp_last30
            FROM ({src}) f
            WHERE f.report_date >= DATE '{prev_start}' AND f.report_date < DATE '{label_end}'
              AND f.gsc_data_available IS TRUE
            GROUP BY f.content_hash_id, f.client_hash_id
            HAVING imp_prev30 >= 100
        )
        SELECT * FROM per_content
    """
    df = con.sql(q).df()
    meta = con.sql(f"""
        SELECT content_hash_id,
               DATEDIFF('day', content_created_date, DATE '{decision_date}') AS content_age_days
        FROM {DIM_CONTENT}
    """).df()
    df = df.merge(meta, on='content_hash_id', how='left')
    df['is_declining'] = (df['imp_last30'] < 0.8 * df['imp_prev30']).astype(int)
    df['pos_prev30'] = df['pos_prev30'].fillna(0)
    df['content_age_days'] = df['content_age_days'].fillna(0)
    df['log_imp_prev30'] = np.log1p(df['imp_prev30'])
    df['log_clk_prev30'] = np.log1p(df['clk_prev30'])
    df['_tie'] = df['content_hash_id'].map(lambda c: int(hashlib.sha256(c.encode()).hexdigest(), 16))
    return df

# Week-5 test rows: features Jan30-Feb28, label March (same window, same rows, same fold rule as Week-5)
march = build_frame('2026-01-30', '2026-03-01', '2026-03-01', '2026-04-01', '2026-03-01', [], ['01', '02', '03'])
# Earlier training window: features December 2025, label January 2026
early_dec = build_frame('2025-12-01', '2026-01-01', '2026-01-01', '2026-02-01', '2026-01-01', ['12'], ['01'])
# Robustness window: features November 2025, label December 2025
early_nov = build_frame('2025-11-01', '2025-12-01', '2025-12-01', '2026-01-01', '2025-12-01', ['11', '12'], [])

def client_fold(client_id, n_folds=5):
    return int(hashlib.sha256(client_id.encode()).hexdigest(), 16) % n_folds
march['fold'] = march['client_hash_id'].map(client_fold)
early_dec['fold'] = early_dec['client_hash_id'].map(client_fold)

FEATURES = ['log_imp_prev30', 'log_clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days']
SEED = 42

def forest():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=SEED)

def precision_at_k(sorted_labels, k):
    head = sorted_labels.head(min(k, len(sorted_labels)))
    return float(head.mean()) if len(head) else float('nan')

def ranked(df, proba):
    return df.assign(score=proba).sort_values(['score', '_tie'], ascending=[False, True]).reset_index(drop=True)

X = march[FEATURES].to_numpy(); y = march['is_declining'].to_numpy()

# BEFORE — Week-5 protocol: grouped by client, train + test inside the SAME window
per_fold, pooled_parts = [], []
for f in sorted(march['fold'].unique()):
    te_mask = march['fold'].to_numpy() == f
    rf = forest().fit(X[~te_mask], y[~te_mask])
    per_fold.append(precision_at_k(ranked(march[te_mask], rf.predict_proba(X[te_mask])[:, 1])['is_declining'], 50))
    pooled_parts.append(march[te_mask].assign(score=rf.predict_proba(X[te_mask])[:, 1]))
before_mean_fold = float(np.mean(per_fold))
before_pool = pd.concat(pooled_parts).sort_values(['score', '_tie'], ascending=[False, True]).reset_index(drop=True)
before_pooled = precision_at_k(before_pool['is_declining'], 50)

# AFTER (time-only) — same forest, trained on an earlier window, all March rows tested at once
rf = forest().fit(early_dec[FEATURES].to_numpy(), early_dec['is_declining'].to_numpy())
proba = rf.predict_proba(X)[:, 1]
after_te = ranked(march, proba)
after_pk = {k: precision_at_k(after_te['is_declining'], k) for k in (20, 50, 100)}
after_auc = roc_auc_score(y, proba)
after_top50_decl = float(after_te['is_declining'].head(50).mean())

# AFTER (time + grouped) — same per-fold protocol as Week-5, only the training period changed
tg_folds, tg_parts = [], []
for f in sorted(march['fold'].unique()):
    tr = early_dec[early_dec['fold'] != f]
    te_mask = march['fold'].to_numpy() == f
    rf = forest().fit(tr[FEATURES].to_numpy(), tr['is_declining'].to_numpy())
    te = ranked(march[te_mask], rf.predict_proba(march[FEATURES].to_numpy()[te_mask])[:, 1])
    tg_folds.append(precision_at_k(te['is_declining'], 50))
    tg_parts.append(te)
tg_mean = float(np.mean(tg_folds))
tg_pool = pd.concat(tg_parts).sort_values(['score', '_tie'], ascending=[False, True]).reset_index(drop=True)
tg_pooled = precision_at_k(tg_pool['is_declining'], 50)

# Robustness — second earlier window (Nov -> Dec), time-only
rf = forest().fit(early_nov[FEATURES].to_numpy(), early_nov['is_declining'].to_numpy())
nov_proba = rf.predict_proba(X)[:, 1]
nov_p50 = precision_at_k(ranked(march, nov_proba)['is_declining'], 50)

print('=== Before vs after — same 81,521 March test rows, same metric, same K, same tie policy ===')
print()
print(f'BEFORE  Week-5 split (grouped by client, same Jan-Feb window)')
print(f'        mean fold P@50 = {before_mean_fold:.3f}   (Week-5 committed receipt: 0.676)')
print(f'        pooled P@50 over all test rows = {before_pooled:.3f}')
print(f'        test base rate = {march["is_declining"].mean():.3f}')
print()
print(f'AFTER   time-aware (train Dec-2025 -> Jan-2026 label, test the same March rows)')
print(f'        all clients at once:  P@20 {after_pk[20]:.2f}  P@50 {after_pk[50]:.2f}  P@100 {after_pk[100]:.2f}   AUC {after_auc:.3f}')
print(f'        top-50 declining share under this model = {after_top50_decl:.2f}  vs base rate {march["is_declining"].mean():.3f}')
print(f'        time + grouped by client (Week-5 fold protocol): mean fold P@50 = {tg_mean:.3f}')
print(f'          folds: ' + ' '.join(f'{v:.2f}' for v in tg_folds) + f'   pooled P@50 = {tg_pooled:.3f}')
print(f'        robustness (train Nov-2025 -> Dec-2025): P@50 = {nov_p50:.2f}')
print()
print('=== Why: the signal is not stable across months (feature vs is_declining correlation r) ===')
short = {'log_imp_prev30': 'log_imp', 'log_clk_prev30': 'log_clk', 'pos_prev30': 'pos',
         'days_with_imp_prev30': 'days', 'content_age_days': 'age'}
print(f'  {"":12s} ' + '  '.join(f'{short[c]:>8}' for c in FEATURES))
for name, w in [('Dec-Jan (train)', early_dec), ('Nov-Dec (train)', early_nov), ('March (test)', march)]:
    print(f'  {name:12s} ' + '  '.join(f'{w[c].corr(w["is_declining"]):+8.3f}' for c in FEATURES))
print()
print(f'  train window: {len(early_dec):,} pages / {early_dec["client_hash_id"].nunique()} clients / base rate {early_dec["is_declining"].mean():.3f}')
print(f'  test window : {len(march):,} pages / {march["client_hash_id"].nunique()} clients / base rate {march["is_declining"].mean():.3f}')

receipt = {
    'snapshot': 'warehouse pulled at run time',
    'before': {
        'mean_fold_P@50': round(before_mean_fold, 4),
        'pooled_P@50': round(before_pooled, 4),
        'week5_receipt_mean_fold_P@50': 0.676,
        'test_base_rate': round(float(march['is_declining'].mean()), 4),
    },
    'after': {
        'time_only': {'P@20': after_pk[20], 'P@50': after_pk[50], 'P@100': after_pk[100], 'AUC': round(after_auc, 4),
                      'top50_declining_share': round(after_top50_decl, 4)},
        'time_plus_grouped': {'mean_fold_P@50': round(tg_mean, 4), 'pooled_P@50': round(tg_pooled, 4),
                              'folds': [round(v, 4) for v in tg_folds]},
        'robustness_nov_dec_P@50': round(nov_p50, 4),
    },
    'correlations_feature_vs_label': {
        'dec_jan_train': {short[c]: round(float(early_dec[c].corr(early_dec['is_declining'])), 4) for c in FEATURES},
        'nov_dec_train': {short[c]: round(float(early_nov[c].corr(early_nov['is_declining'])), 4) for c in FEATURES},
        'march_test': {short[c]: round(float(march[c].corr(march['is_declining'])), 4) for c in FEATURES},
    },
}
rec_path = os.path.join(OUT_DIR, 'w06_validation_audit_receipt.json')
with open(rec_path, 'w') as fh:
    json.dump(receipt, fh, indent=2)
print()
print(f'Receipt: {rec_path}')

=== Before vs after — same 81,521 March test rows, same metric, same K, same tie policy ===

BEFORE  Week-5 split (grouped by client, same Jan-Feb window)
        mean fold P@50 = 0.672   (Week-5 committed receipt: 0.676)
        pooled P@50 over all test rows = 0.560
        test base rate = 0.249

AFTER   time-aware (train Dec-2025 -> Jan-2026 label, test the same March rows)
        all clients at once:  P@20 0.10  P@50 0.10  P@100 0.14   AUC 0.525
        top-50 declining share under this model = 0.10  vs base rate 0.249
        time + grouped by client (Week-5 fold protocol): mean fold P@50 = 0.384
          folds: 0.24 0.08 1.00 0.42 0.18   pooled P@50 = 0.200
        robustness (train Nov-2025 -> Dec-2025): P@50 = 0.28

=== Why: the signal is not stable across months (feature vs is_declining correlation r) ===
                log_imp   log_clk       pos      days       age
  Dec-Jan (train)   +0.058    -0.060    +0.133    -0.018    +0.076
  Nov-Dec (train)   +0.089    +0.072    

## 3. Leakage audit

The same hunt as Week 3, on the final Week-5 feature set. Three checks, run below.

**1. Denominator leak (the important one).** `log_imp_prev30` is the *denominator* of my label — `is_declining` is defined as `imp_last30 < 0.8 * imp_prev30` — so this feature is label-derived by definition, even though it is measured before the label window (taxonomy type 1: a transformed copy of the label's own inputs). The test: train the same forest **with** and **without** `log_imp_prev30`, same folds, same seed, same everything else. If removing it collapses P@50, the model's edge is carried by the leak; if the score holds, the pattern lives in the other features too and the leak is definitional but not load-bearing. I also run two controls: dropping `log_clk_prev30` (a sibling click-count feature) and dropping both count features.

**2. Window check.** Every feature must be computed strictly before the label window (no `report_date >= 2026-03-01` feeds a feature) and the feature and label windows must not overlap.

**3. Product-flag check.** None of the features are FlyRank product scores (health, optimization flags, composite scores) or future signals — the set is GSC aggregates plus content age.

**Measured reading** (numbers in the cell below): removing the denominator keeps mean fold P@50 within noise of the full set (no collapse), and dropping both count features keeps it ≈0.65. So the leak is real in definition but the Week-5 edge is **not** carried by the denominator alone — the same-window ranking would not vanish if that feature were removed. What *does* change the result is time (Section 2), not this feature.

In [3]:
# Section 3 — leakage audit on the final Week-5 feature set
variants = {
    'all 5 features (Week-5 set)': FEATURES,
    'no log_imp_prev30 (label denominator)': [c for c in FEATURES if c != 'log_imp_prev30'],
    'no log_clk_prev30 (sibling control)': [c for c in FEATURES if c != 'log_clk_prev30'],
    'no both count features': [c for c in FEATURES if c not in ('log_imp_prev30', 'log_clk_prev30')],
}

rows = []
for name, cols in variants.items():
    Xs = march[cols].to_numpy()
    vals = []
    for f in sorted(march['fold'].unique()):
        te_mask = march['fold'].to_numpy() == f
        rf = forest().fit(Xs[~te_mask], y[~te_mask])
        vals.append(precision_at_k(ranked(march[te_mask], rf.predict_proba(Xs[te_mask])[:, 1])['is_declining'], 50))
    rows.append({'variant': name, 'mean_fold_P@50': round(float(np.mean(vals)), 4),
                 'folds': [round(v, 3) for v in vals]})

print('Mean fold P@50 by feature set — same folds, same seed, same forest (300 trees):')
print(pd.DataFrame(rows).to_string(index=False))
print()

# 1. Denominator leak conclusion
full = rows[0]['mean_fold_P@50']
no_denom = rows[1]['mean_fold_P@50']
print(f'Denominator leak test: full {full:.3f}  vs  without log_imp_prev30 {no_denom:.3f} '
      f'-> change of {full - no_denom:+.3f} (no collapse)')
print()

# 2. Window check — feature window is closed before the label window starts
print('Window check:')
print('  feature window  = report_date in [Jan 30, Mar 1)  (imp_prev30 / clk / pos / days)')
print('  label window    = report_date in [Mar 1, Apr 1)   (imp_last30)')
print('  label field in the feature set?', 'imp_last30' in FEATURES,
      '| feature window overlaps label window? False (enforced by the SQL CASE WHEN)')
print()

# 3. Product-flag check — the five features are GSC aggregates + content age
external = [c for c in FEATURES if c not in ('log_imp_prev30', 'log_clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days')]
print('Product-flag check: non-GSC, non-age fields in the feature set?', external)
print('  -> no health score, no optimization flags, no composite scores, no future signals.')

Mean fold P@50 by feature set — same folds, same seed, same forest (300 trees):
                              variant  mean_fold_P@50                         folds
          all 5 features (Week-5 set)           0.672   [0.5, 0.98, 1.0, 0.6, 0.28]
no log_imp_prev30 (label denominator)           0.680 [0.48, 0.96, 1.0, 0.54, 0.42]
  no log_clk_prev30 (sibling control)           0.672  [0.44, 0.9, 1.0, 0.58, 0.44]
               no both count features           0.644   [0.5, 0.9, 0.98, 0.5, 0.34]

Denominator leak test: full 0.672  vs  without log_imp_prev30 0.680 -> change of -0.008 (no collapse)

Window check:
  feature window  = report_date in [Jan 30, Mar 1)  (imp_prev30 / clk / pos / days)
  label window    = report_date in [Mar 1, Apr 1)   (imp_last30)
  label field in the feature set? False | feature window overlaps label window? False (enforced by the SQL CASE WHEN)

Product-flag check: non-GSC, non-age fields in the feature set? []
  -> no health score, no optimization flags, no

## 4. Claim rewrite

My boldest Week-5 sentence: **"My model predicts which pages will decline."**

Too bold for three *measured* reasons:

1. **The split never saw a future period.** Week-5 grouped by client but trained and tested inside the same calendar window. This audit shows the model does not transfer forward: P@50 drops from ≈0.67 (same window) to ≈0.35–0.40 (unseen clients, earlier training window) or ≈0.1–0.2 (unseen time, all clients).
2. **The signal is not stable across months.** Feature–label correlations flip sign between windows, so "predicts decline" overstates a pattern that is window-local.
3. **A label-derived feature is in the set.** `log_imp_prev30` is the denominator the label is built from — definitionally label-derived — so any same-window score leans on a feature the label itself is constructed from (the audit shows it is not the load-bearing one, but it is still definitionally a leak).

Safe rewrite (what I can honestly say):

> On the measured Week-5 data, the model produces a **directional ranking** of pages by their **observed** decline signal. It is **decision-support** — it helps a reviewer decide which pages to open first — not a prediction of which pages will decline. The ranking does not transfer to a new calendar period (P@50 falls from ≈0.67 to ≈0.1–0.4), so it is not a forward predictor.

Every word is now tied to a measured claim: *directional* (a ranking, not a cause), *observed* (from GSC history before the label window), *measured* (on a defined split), *decision-support* (it prioritises which pages a reviewer opens), and the forward-prediction limit is stated as a number.

In [4]:
# Section 4 — the two sentences side by side, plus a word check
original = 'My model predicts which pages will decline.'
rewritten = ('On the measured Week-5 data, the model produces a directional ranking of pages '
             'by their observed decline signal. It is decision-support - it helps a reviewer '
             'decide which pages to open first - not a prediction of which pages will decline. '
             'The ranking does not transfer to a new calendar period (P@50 falls from ~0.67 '
             'to ~0.1-0.4), so it is not a forward predictor.')

print('Original (too bold):', original)
print()
print('Rewritten (safe):', rewritten)
print()
safe_words = ['observed', 'measured', 'directional', 'decision-support', 'not a prediction']
print('Safe words present in the rewrite:', {w: (w in rewritten) for w in safe_words})
assert all(w in rewritten for w in safe_words)
print()
print('Measured anchors used in the rewrite: same-window P@50 ~0.67, time-aware P@50 ~0.1-0.4 (Section 2).')

Original (too bold): My model predicts which pages will decline.

Rewritten (safe): On the measured Week-5 data, the model produces a directional ranking of pages by their observed decline signal. It is decision-support - it helps a reviewer decide which pages to open first - not a prediction of which pages will decline. The ranking does not transfer to a new calendar period (P@50 falls from ~0.67 to ~0.1-0.4), so it is not a forward predictor.

Safe words present in the rewrite: {'observed': True, 'measured': True, 'directional': True, 'decision-support': True, 'not a prediction': True}

Measured anchors used in the rewrite: same-window P@50 ~0.67, time-aware P@50 ~0.1-0.4 (Section 2).


## Self-check

Before I submit, each line confirmed:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed via nbclient, output verified)
- [x] No client names, URLs, or private queries anywhere — hash IDs only (`client_hash_id`, `content_hash_id`), and the SQL shows only table/field names and date ranges, no private filter values
- [x] My claims use careful words: observed, measured, directional, decision-support — and the forward-prediction limit is stated as a number
- [x] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card. Done.